# Bruno 的第 1 周个性化技术导师

## 练习目标（理念）

结合 **OpenAI** 云端模型与本地 **Ollama**，用**个性化 system prompt** 和少量 **few-shot 示例**，搭建一位友好、简洁的技术问答导师。

- **输入**：一个技术问题（可改写成你自己的问题）
- **输出**：按 Bruno 偏好风格给出的解释（先实用含义、再适量细节）
- **对比**：同一套 `messages` 分别打给 `gpt-4o-mini`（流式）和 `llama3.2`（非流式）

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `openai.chat.completions.create(...)` |
| `messages`（system / user / assistant） | 个性化 system + few-shot 正反例 + 真问题 |
| 流式输出 `stream=True` | GPT 一格边收边 `print` |
| Ollama（OpenAI 兼容） | `base_url=http://localhost:11434/v1` |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `OPENAI_API_KEY`；本地需已 `ollama pull llama3.2` 并启动 Ollama
3. 在「提问」单元格改写 `question`，再分别跑 GPT 与 Llama 两格，对比回答风格


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os

# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：同一套 SDK 既能打云端，也能打 Ollama 的 OpenAI 兼容端点
from openai import OpenAI


In [ ]:
# ========== 常量：模型名与本地地址集中写在一处 ==========

# OpenAI 云端小模型：便宜、够用，适合做解释类问答
MODEL_GPT = "gpt-4o-mini"
# 本地 Ollama 模型名：需事先 ollama pull llama3.2；字符串必须和本机已安装的模型名一致
MODEL_LLAMA = "llama3.2"
# Ollama 的 OpenAI 兼容 API 根地址（注意是 /v1，不是原生 /api/chat）
OLLAMA_BASE_URL = "http://localhost:11434/v1"


In [ ]:
# ========== 环境 + 双客户端：云端 OpenAI 与本地 Ollama ==========

# 加载 .env；override=True 表示：即使进程里已有同名环境变量，也用 .env 里的值覆盖
load_dotenv(override=True)

# 从环境变量取出 OpenAI 密钥
api_key = os.getenv("OPENAI_API_KEY")
# 没有密钥就立刻失败，避免后面调用时才报含糊错误（错误文案保持英文原样）
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is missing. Add it to your .env file.")

# 云端客户端：默认从环境变量 OPENAI_API_KEY 读密钥
openai = OpenAI()
# 本地客户端：指向 Ollama 的 OpenAI 兼容端点；api_key 对本地通常只是占位字符串
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")


## 提出一个技术问题

把下面单元格里的 `question` 改成你想问的内容；课程期间可以把这个笔记本当成随身学习工具反复用。


In [ ]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# user 侧的具体问题：三引号字符串；发给模型的内容保持英文（可运行 / 影响回答的字符串不翻译）
question = """
Please explain what is the best AI workflow to use with Godot to make a retro style rpg game with pixel art
"""


In [ ]:
# ========== 个性化导师提示词 + few-shot 示例，组装 messages ==========

# system prompt：定角色、学习者画像、回答规范（发给模型的指令，保留英文，改译会改变行为）
system_prompt = """
You are Bruno's personal AI Engineering tutor.

Bruno likes answers that are friendly, direct, and succinct. He is learning AI engineering,
works with PHP, enjoys game development with Godot, and wants to learn more DevOps.

How to answer:
- Start with the practical meaning first.
- Keep the answer concise, but include enough detail to be useful.
- Use small code examples only when they clarify the idea.
- Mention Godot/game development, PHP, or DevOps only when the analogy genuinely helps.
- Point out common gotchas or edge cases.
- If the question is ambiguous, state the most likely interpretation and continue.
- Use markdown, but do not wrap the entire answer in a code block.
"""

# few-shot 正例：示范「好问题」长什么样（字符串内容不翻译）
example_question = """
What is a generator in Python?
"""

# few-shot 正例：示范「好回答」——先实用含义、再小例子、再场景类比
example_answer = """
A generator is a function or expression that produces values one at a time instead of building
a full list in memory.

The key idea is lazy execution: Python pauses the generator after each `yield` and resumes it
when the next value is requested.

Small example:

```python
def numbers():
    yield 1
    yield 2

for number in numbers():
    print(number)
```

This is useful when the sequence could be large, like streaming logs in a DevOps script or
loading game assets step by step instead of all at once.
"""

# few-shot 反例：故意写得太抽象、术语堆砌，后面让模型「学会避开」
bad_answer_example = """
Generators are special iterable coroutine-like lazy objects that implement iterator protocols
and facilitate deferred execution semantics.
"""

# messages：system → 正例问答 → 反例纠正 → 真正的用户问题（多轮上下文一起发给模型）
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": example_question},
    {"role": "assistant", "content": example_answer},
    {
        "role": "user",
        # 把反例拼进 user 消息，明确告诉模型「不要这样答」
        "content": "Avoid answers like this because they are too abstract and jargon-heavy:\n"
        + bad_answer_example,
    },
    {
        "role": "assistant",
        # 助手确认：后续会用朴实、实用的风格（仍是发给模型的英文）
        "content": "Understood. I will explain concepts plainly and practically, with just enough technical detail.",
    },
    # 最后一轮 user：真正要问的问题（上面单元格里的 question）
    {"role": "user", "content": question},
]


In [ ]:
# ========== 调用 gpt-4o-mini：流式边收边打印 ==========

# chat.completions.create：发起 Chat Completions；stream=True 表示持续返回增量 chunk
stream = openai.chat.completions.create(
    # 使用上面定义的云端模型常量
    model=MODEL_GPT,
    # 把组装好的多轮 messages 整包发给模型
    messages=messages,
    # stream=True：不要等整段生成完，而是持续返回增量 delta
    stream=True,
)

# 逐块遍历流式事件
for chunk in stream:
    # 增量文本在 choices[0].delta.content；结束或空块时可能为 None
    content = chunk.choices[0].delta.content
    # 有文本才打印；end="" 不换行，flush=True 立刻刷到终端（打字机效果）
    if content:
        print(content, end="", flush=True)


In [ ]:
# ========== 调用本地 Llama 3.2：非流式一次取完整回答 ==========

# 走 ollama 客户端（OpenAI 兼容端点）；不传 stream，默认等整段生成完再返回
response = ollama.chat.completions.create(
    # 本地模型名必须和 ollama list / pull 的名字一致
    model=MODEL_LLAMA,
    # 与 GPT 共用同一套 messages，便于对比风格差异
    messages=messages,
)

# 非流式：完整回答在 choices[0].message.content
print(response.choices[0].message.content)


## 补充说明

想试其他本地模型时，先用 Ollama `pull` 下来，再改常量 `MODEL_LLAMA`。机器内存较小时，可考虑 `llama3.2:1b` 这类更小的变体。
